# small\_fable — INTERLEAVED (agentic) SFT + planner-only RL

Closed-loop, turn-by-turn: the planner emits a turn's primitives (conditioned on the reasoning so
far), the executor writes that turn's response, repeat — so each response chunk is **gated by its
primitives** and the plan is **load-bearing by construction**. See
[`ARCHITECTURE_INTERLEAVED.md`](https://github.com/sinha-k-prat/small_fable/blob/main/ARCHITECTURE_INTERLEAVED.md).

- **SFT** (`--interleaved`): one interleaved stream/trace; reports `ablation_gap` (drop plan) AND
  `shuffle_gap` (random primitives) — the stronger detector.
- **RL** (`--interleaved --freeze_executor`): **only the planner head + plan_emb train** (LoRA frozen,
  `lam_resp=0`) — all reward improvement flows through plan choice.
- Train on sets 1000+2000; **hold out set 3000 (flipped-answer)** as the unseen generalization test.
- `run_interleaved` prints **each turn as it decodes** (transparent).

> T4 GPU + a Hugging Face **write** token as Colab secret `HF_TOKEN`. Then `Runtime → Run all`.
> Crash-safe: every stage checkpoints to HF every ~10 min and `--resume`; just Run-all again.


## 0 · GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU — Runtime → Change runtime type → T4 GPU, then re-run.'


## 1 · Clone & install

In [ ]:
import os
REPO = 'https://github.com/sinha-k-prat/small_fable.git'
if not os.path.isdir('small_fable'):
    !git clone -q $REPO
else:
    !cd small_fable && git pull -q
%cd small_fable
!pip install -q -r requirements.txt huggingface_hub
!pip uninstall -y torchao >/dev/null 2>&1; echo 'torchao removed (peft clash workaround)'
print('\nReady.')


## 2 · Hugging Face setup

In [ ]:
import os
from huggingface_hub import HfApi, create_repo, snapshot_download, login
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from huggingface_hub import notebook_login; notebook_login()
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False); os.environ['HF_TOKEN'] = HF_TOKEN
api = HfApi()
HF_REPO = f"{api.whoami()['name']}/small_fable-planner"
create_repo(HF_REPO, repo_type='model', exist_ok=True, private=False)
print('HF repo:', f'https://huggingface.co/{HF_REPO}')
def pull_ckpt(sub):
    try:
        snapshot_download(repo_id=HF_REPO, allow_patterns=[f'{sub}/*'], local_dir='.')
        print(f'[pull] {sub}: ' + ('resumable' if os.path.exists(f'{sub}/train_state.pt') else 'fresh'))
    except Exception as e:
        print(f'[pull] {sub}: nothing to resume ({e})')
def push_file(p):
    try: api.upload_file(path_or_fileobj=p, path_in_repo=os.path.basename(p), repo_id=HF_REPO,
                         commit_message=f'upload {p}'); print('[hf] pushed', p)
    except Exception as e: print('[hf] push skipped:', e)


## 2.5 · Stage 0 — build the INTERLEAVED corpus (fast, CPU)
`--interleaved` keeps the per-turn structure (`turns:[{plan,response}]`) and writes a `plan_vocab.json`
with `BOP`/`FINALIZE_ALL` markers. Train = sets 1000+2000; held-out generalization = set 3000.

In [ ]:
import json
TR = 'traces'; DATA = 'dataset/traces_sft.jsonl'
# interleaved vocab from ALL three sets (so every token incl. eval-only ones is covered):
!python -u traces_to_sft.py --interleaved --traces $TR/hard_reasoning_traces_1000.jsonl $TR/hard_reasoning_traces_2000.jsonl $TR/hard_reasoning_traces_3000.jsonl --answers $TR/answers_1000.jsonl $TR/answers_2000.jsonl $TR/answers_3000.jsonl --out /tmp/_all.jsonl --vocab_out plan_vocab.json
# train corpus = sets 1000 + 2000:
!python -u traces_to_sft.py --interleaved --traces $TR/hard_reasoning_traces_1000.jsonl $TR/hard_reasoning_traces_2000.jsonl --answers $TR/answers_1000.jsonl $TR/answers_2000.jsonl --out $DATA --vocab_out /tmp/_v.json
# held-out generalization = set 3000 (flipped):
!python -u traces_to_sft.py --interleaved --traces $TR/hard_reasoning_traces_3000.jsonl --answers $TR/answers_3000.jsonl --out dataset/traces_sft_3000.jsonl --vocab_out /tmp/_v.json
push_file('plan_vocab.json')
N = sum(1 for _ in open(DATA))
HELD = 60; TRAIN = min(N - HELD, 800); ROLL_TRAIN = min(TRAIN, 120)
MAX_RESP = 256; MAX_TURNS = 6; MAX_PLAN = 12
v = json.load(open('plan_vocab.json'))
print('interleaved vocab:', len(v['vocab']), 'tokens | markers:', v.get('markers'), '| interleaved:', v.get('interleaved'))
print(f'train(1000+2000) rows={N}  ->  TRAIN={TRAIN} HELD={HELD} ROLL_TRAIN={ROLL_TRAIN} MAX_RESP={MAX_RESP} MAX_TURNS={MAX_TURNS}')


## 3 · Stage 1 — Interleaved SFT  *(checkpoints + resumes via HF)*
Each turn's response is generated conditioned on that turn's primitives. Watch the held line:
`ablation_gap` (drop plan) and `shuffle_gap` (random primitives) should now be **positive** — the
executor genuinely needs the plan.

In [ ]:
pull_ckpt('joint_ckpt')
!python -u train_sft.py --interleaved --data $DATA --train $TRAIN --held $HELD --epochs 4 --lr 2e-5 --bs 2 --max_resp $MAX_RESP --max_turns $MAX_TURNS --max_plan $MAX_PLAN --probe 0 --out joint_ckpt --device cuda --resume --hf_repo $HF_REPO --ckpt_every_min 10


## 3.5 · Loss table (per epoch)
`held_ablation_gap` and `held_shuffle_gap` green = positive = plan load-bearing. Re-run anytime.

In [ ]:
import json, os, pandas as pd
from IPython.display import display
if os.path.exists('sft_metrics.jsonl'):
    df = pd.DataFrame([json.loads(l) for l in open('sft_metrics.jsonl')]).drop_duplicates('epoch', keep='last')
    gaps = [x for x in df.columns if 'gap' in x]; ces = [x for x in df.columns if x.endswith('_ce') or x == 'kl']
    sty = df.style.hide(axis='index').format(precision=3).set_caption('Interleaved SFT — per-epoch (gap > 0 = plan load-bearing)')
    try:
        if gaps: sty = sty.background_gradient(subset=gaps, cmap='RdYlGn')
        if ces:  sty = sty.background_gradient(subset=ces, cmap='RdYlGn_r')
    except Exception: pass
    display(sty)
else:
    print('no sft_metrics.jsonl yet — run Stage 1 first.')


## 4 · Stage 2a — Interleaved rollouts

In [ ]:
pull_ckpt('joint_ckpt')
!python -u rollout_offline.py --interleaved --sft_ckpt joint_ckpt --data $DATA --train $ROLL_TRAIN --group 8 --temp 1.2 --top_p 0.98 --max_resp $MAX_RESP --max_turns $MAX_TURNS --out rl_rollouts.jsonl --device cuda
push_file('rl_rollouts.jsonl')


## 5 · Stage 2b — Interleaved GRPO, **planner-only**  *(--freeze_executor)*
LoRA frozen, `lam_resp=0`: **only the planner head + plan_emb train**. All reward improvement flows
through plan choice — the closed-loop planner learns to steer the (fixed) executor.

In [ ]:
pull_ckpt('joint_ckpt'); pull_ckpt('rl_ckpt')
import os; assert os.path.exists('rl_rollouts.jsonl'), 'run Stage 2a first'
!python -u train_grpo_offline.py --interleaved --freeze_executor --rollouts rl_rollouts.jsonl --sft_ckpt joint_ckpt --data $DATA --out rl_ckpt --inner_epochs 3 --lr 1e-4 --clip_eps 0.2 --beta_plan 1.0 --beta_ce 0.1 --held 16 --max_resp $MAX_RESP --device cuda --resume --hf_repo $HF_REPO --ckpt_every_min 10
print('All checkpoints on HF:', f'https://huggingface.co/{HF_REPO}/tree/main')


## 6 · In-distribution eval (interleaved): SFT vs planner-only-RL
_Slow — closed-loop decode ×3 conditions per row._

In [ ]:
import torch
from model_joint import JointModel
from train_sft import eval_held_interleaved, load_rows
held_rows = load_rows(DATA)[TRAIN:TRAIN+HELD]
for ckpt, label in [('joint_ckpt', 'SFT'), ('rl_ckpt', 'SFT + planner-only RL')]:
    m = JointModel.from_checkpoint('Qwen/Qwen2.5-1.5B-Instruct', ckpt, device='cuda'); m.eval()
    r = eval_held_interleaved(m, held_rows, max_turns=MAX_TURNS, max_plan=MAX_PLAN, max_resp=MAX_RESP, sample=True)
    print(f'[{label}] in-dist held:', {k: round(v, 3) for k, v in r.items()})
    del m; torch.cuda.empty_cache()


## 7 · Generalization — set 3000 (flipped-answer, NEVER trained)
The headline: trained on 1000+2000, evaluated on 3000 where the answer flips. Accuracy holding =
reasoning, not memorizing.

In [ ]:
import torch
from model_joint import JointModel
from train_sft import eval_held_interleaved, load_rows
gen_rows = load_rows('dataset/traces_sft_3000.jsonl')[:60]
for ckpt, label in [('joint_ckpt', 'SFT'), ('rl_ckpt', 'SFT + planner-only RL')]:
    m = JointModel.from_checkpoint('Qwen/Qwen2.5-1.5B-Instruct', ckpt, device='cuda'); m.eval()
    r = eval_held_interleaved(m, gen_rows, max_turns=MAX_TURNS, max_plan=MAX_PLAN, max_resp=MAX_RESP, sample=True)
    print(f'[{label}] SET 3000 (unseen, flipped):', {k: round(v, 3) for k, v in r.items()})
    del m; torch.cuda.empty_cache()


## 8 · Head-to-head on one hard prompt — transparent per-turn decode

In [ ]:
import torch
from model_joint import JointModel
HARD_PROMPT = ('A snail is at the bottom of a 12-meter well. Each day it climbs 4 meters, '
               'but each night it slides back 3 meters. On which day does it first reach the top?')
for ckpt, label in [('joint_ckpt', 'SFT only'), ('rl_ckpt', 'SFT + planner-only GRPO')]:
    m = JointModel.from_checkpoint('Qwen/Qwen2.5-1.5B-Instruct', ckpt, device='cuda'); m.eval()
    print(f'\n{"="*72}\n{label}  ({ckpt})\n{"="*72}')
    p_ids, p_attn = m.batch_prompts([HARD_PROMPT])
    rec = m.run_interleaved(p_ids[0], p_attn[0], sample=True, temp=0.7,
                            max_turns=MAX_TURNS, max_plan=MAX_PLAN, max_resp=MAX_RESP, verbose=True)
    print('FINAL:', m.interleaved_answer_text(rec)[:400])
    del m; torch.cuda.empty_cache()
